In [38]:
import gcamreader
import os
import sys
import subprocess
import pandas as pd
import pickle
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom
from itertools import product
# Get the parent directory of the current notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Add the parent directory to sys.path
sys.path.append(parent_dir)
from utils import catTech, string_to_xml_file, custom_colors, stack_order, ej_to_twh, convert_to_mt, classify_ghg, gwpAr5

# Korea's Regulation to Promote Zero Energy Buildings

A ZEB Regulation adopted for new buildings. Public buildings should achieve energy self-sufficiency rate(ESSR) at least 20% for 2023, 40% for 2025, 60% for 2030. For year 2020, this regulation was applied only for buildings with floorspace more than 500m^2. Scenario adopts ZEB from year 2025 and we assume that after 2030, 60% requirement is applied. Private buildings with floorspace more than 1,000 m^2 should achieve 20% of ESSR in 2025. After 2030 the requirement is applied to ones with floor space more than 500 m^2.

We model ZEB regulation by two steps: 1. calculate the total floor space required to attain full energy self-sufficiency. For example if a building with floor space 1000 m^2 attains 30% of ESSR, then we regard a building with floor space 300 m^2 attains full sufficiency. 2. We use unit energy consumption EJ/m^2 to calculate how much energy is needed. In GCAM, we assume energy is supplied by rooftop PV technology.

We refer two external dataset `/ext/240417(별첨)_23년_기준_건축_인허가_통계(건축정책과).xlsx` and `ext/240417(별첨)_23년_기준_건축물_현황_통계(건축정책과).xlsx` for estimation.

## Step 1 ZEB Buildings Share

In [8]:
tot_pr_floor_space = 4227660684.322 - 376645408.432
new_pr_floor_space = 147394196.42337 - 9195076.7802
new_pr_ratio = new_pr_floor_space / tot_pr_floor_space
new_pr_ratio

0.035886411697297435

In [ ]:
fs_gt_1000_share = 0.07369
fs_gt_500_share = 0.15460
ss_ratio_g5 = 0.2
new_pr_ratio_ss_25 = fs_gt_1000_share * new_pr_ratio * ss_ratio_g5 * 5
new_pr_ratio_ss_30 = new_pr_ratio_ss_25 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
new_pr_ratio_ss_35 = new_pr_ratio_ss_30 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
new_pr_ratio_ss_40 = new_pr_ratio_ss_35 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_g5 * 5
print(new_pr_ratio_ss_25, new_pr_ratio_ss_30, new_pr_ratio_ss_35, new_pr_ratio_ss_40)

0.002644469677973848 0.00819250892637603 0.013740548174778214 0.019288587423180396


In [ ]:
fs_gt_500_share = 0.15460
ss_ratio = 0.2
private_vals = [fs_gt_1000_share * new_pr_ratio, fs_gt_500_share * ss_ratio, fs_gt_500_share * ss_ratio, fs_gt_500_share * ss_ratio]

In [10]:
tot_public_floor_space = 376645408.432
new_public_floor_space = 9195076.7802
new_public_ratio = new_public_floor_space / tot_public_floor_space
new_public_ratio

0.0244130860866716

In [12]:
ss_ratio_25 = 0.4
ss_ratio_30 = 0.6
new_public_ratio_ss_25 = fs_gt_500_share * new_public_ratio * ss_ratio_25 * 5
new_public_ratio_ss_30 = new_public_ratio_ss_25 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_35 = new_public_ratio_ss_30 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_40 = new_public_ratio_ss_35 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
print(new_public_ratio_ss_25, new_public_ratio_ss_30, new_public_ratio_ss_35, new_public_ratio_ss_40)

0.007548526217998858 0.018871315544997148 0.030194104871995437 0.04151689419899372


In [12]:
public_vals = [ss_ratio_25, ss_ratio_30, ss_ratio_30, ss_ratio_30]

In [13]:
pr_share = 0.911
serZebShare = pd.Series([
    new_pr_ratio_ss_25 * pr_share + new_public_ratio_ss_25 * (1-pr_share),
    new_pr_ratio_ss_30 * pr_share + new_public_ratio_ss_30 * (1-pr_share),
    new_pr_ratio_ss_35 * pr_share + new_public_ratio_ss_35 * (1-pr_share),
    new_pr_ratio_ss_40 * pr_share + new_public_ratio_ss_40 * (1-pr_share),
], index=[2025, 2030, 2035, 2040])
serZebShare

2025    0.003081
2030    0.009143
2035    0.015205
2040    0.021267
dtype: float64

## Unit Energy Demand

In [14]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [15]:
db_file = "database_basexdb"
query_file = db_path / "queries" / "Main_queries.xml"
conn = gcamreader.LocalDBConn(db_path, db_file)
queries = gcamreader.parse_batch_query(query_file)

Database scenarios: Reference


In [16]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [17]:
q = queries[70]
print(q.title)
dfFloor = conn.runQuery(query=q, scenarios=['Reference'], regions=['South Korea'])
dfFloor['scenario'] = dfFloor['scenario'].str.split(',').str[0]
dfFloor.head()

building floorspace


,Units,scenario,region,building,nodeInput,building-node-input,Year,value
0,billion m^2,Reference,South Korea,comm,comm,comm_building,1975,0.333635
1,billion m^2,Reference,South Korea,comm,comm,comm_building,1990,0.489230
2,billion m^2,Reference,South Korea,comm,comm,comm_building,2005,0.677551
3,billion m^2,Reference,South Korea,comm,comm,comm_building,2010,0.689301
4,billion m^2,Reference,South Korea,comm,comm,comm_building,2015,0.707074


In [18]:
q = queries[77]
print(q.title)
dfEnergy = conn.runQuery(query=q, scenarios=['Reference'], regions=['South Korea'])
dfEnergy['nodeInput'] = dfEnergy['sector'].apply(lambda s: 'comm' if s.startswith('comm') else 'resid')
dfEnergy['scenario'] = dfEnergy['scenario'].str.split(',').str[0]
dfEnergy.head()

building final energy by service and fuel


,Units,scenario,region,sector,input,Year,value,nodeInput
0,EJ,Reference,South Korea,comm cooling,delivered gas,1990,0.000386,comm
1,EJ,Reference,South Korea,comm cooling,delivered gas,2005,0.008490,comm
2,EJ,Reference,South Korea,comm cooling,delivered gas,2010,0.009812,comm
3,EJ,Reference,South Korea,comm cooling,delivered gas,2015,0.010216,comm
4,EJ,Reference,South Korea,comm cooling,delivered gas,2020,0.009217,comm


In [19]:
# final energy consumption by floorspace: comm
denon = dfFloor[(dfFloor['Year'] >= 2025) & (dfFloor['Year'] <= 2040)].groupby(['scenario', 'region', 'Year'])['value'].sum()
num = dfEnergy[(dfEnergy['Year'] >= 2025) & (dfEnergy['Year'] <= 2040)].groupby(['scenario', 'region', 'Year'])['value'].sum()
enConUnit = num / denon
enConUnit.index = range(2025, 2045, 5)
enConUnit

2025    0.882254
2030    0.895309
2035    0.897442
2040    0.892007
Name: value, dtype: float64

## 3. Floor Space Tot

In [20]:
serFloor = dfFloor[dfFloor['Year'].isin([2025, 2030, 2035, 2040])].groupby(['Year'])['value'].sum()
serFloor

Year
2025    2.079178
2030    2.150664
2035    2.193658
2040    2.209648
Name: value, dtype: float64

In [21]:
zebTarget = serFloor * serZebShare * enConUnit
zebTarget

Year
2025    0.005652
2030    0.017605
2035    0.029934
2040    0.041918
dtype: float64

## Rooftop PV Generation in the Reference Scenario

In [22]:
q = queries[9]
print(q.title)
dfGen = conn.runQuery(query=q, scenarios=['Reference'], regions=['South Korea'])
dfGen['scenario'] = dfGen['scenario'].str.split(',').str[0]
dfGen.head()

elec gen by gen tech


,Units,scenario,region,subsector,technology,output,Year,value
0,EJ,Reference,South Korea,biomass,biomass (IGCC),electricity,2025,0.000417
1,EJ,Reference,South Korea,biomass,biomass (IGCC),electricity,2030,0.001082
2,EJ,Reference,South Korea,biomass,biomass (IGCC),electricity,2035,0.002026
3,EJ,Reference,South Korea,biomass,biomass (IGCC),electricity,2040,0.003233
4,EJ,Reference,South Korea,biomass,biomass (IGCC),electricity,2045,0.004462


In [23]:
dfGen['technology'].unique()

array(['biomass (IGCC)', 'biomass (conv)', 'coal (IGCC)',
       'coal (conv pul)', 'gas (CC)', 'gas (steam/CT)', 'hydro',
       'Gen_III', 'Gen_II_LWR', 'refined liquids (CC)',
       'refined liquids (steam/CT)', 'rooftop_pv', 'CSP_storage', 'PV',
       'PV_storage', 'wind', 'wind_offshore', 'wind_storage'],
      dtype=object)

In [24]:
serRefGen = dfGen[(dfGen['Year'] >= 2020) & (dfGen['Year'] <= 2040) & (dfGen['technology'] == 'rooftop_pv')].groupby(['Year', 'scenario'])['value'].sum()
serRefGen.index = [2020, 2025, 2030, 2035, 2040]
serRefGen

2020    0.002308
2025    0.017475
2030    0.050636
2035    0.085309
2040    0.103447
Name: value, dtype: float64

In [25]:
roofTopGen = zebTarget + serRefGen
roofTopGen

2020         NaN
2025    0.023126
2030    0.068241
2035    0.115243
2040    0.145365
dtype: float64

We add the rooptop generation of South Korea in 2020 to the constraint.

```xml
<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="ZEB-Certification-Target">
        <policyType>subsidy</policyType>
        <market>South Korea</market>
        <constraint year="2020">0.012</constraint>
        <constraint year="2025">0.030</constraint>
        <constraint year="2030">0.086</constraint>
        <constraint year="2035">0.144</constraint>
        <constraint year="2040">0.185</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>

```

## Enhanced Policy

Private buildings are regulated by the same condition for the public buildings after 2030

In [26]:
tot_pr_floor_space = 4227660684.322 - 376645408.432
new_pr_floor_space = 147394196.42337 - 9195076.7802
new_pr_ratio = new_pr_floor_space / tot_pr_floor_space
new_pr_ratio

0.035886411697297435

In [27]:
fs_gt_1000_share = 0.07369
fs_gt_500_share = 0.15460
ss_ratio = 0.2
ss_ratio_30 = 0.4
new_pr_ratio_ss_25 = fs_gt_1000_share * new_pr_ratio * ss_ratio * 5
new_pr_ratio_ss_30 = new_pr_ratio_ss_25 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
new_pr_ratio_ss_35 = new_pr_ratio_ss_30 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
new_pr_ratio_ss_40 = new_pr_ratio_ss_35 + (new_pr_ratio * fs_gt_500_share) * ss_ratio_30 * 5
print(new_pr_ratio_ss_25, new_pr_ratio_ss_30, new_pr_ratio_ss_35, new_pr_ratio_ss_40)

0.002644469677973848 0.013740548174778214 0.024836626671582582 0.035932705168386946


In [28]:
private_vals += [fs_gt_1000_share * new_pr_ratio, fs_gt_500_share * ss_ratio_30, fs_gt_500_share * ss_ratio_30, fs_gt_500_share * ss_ratio_30]

NameError: name 'private_vals' is not defined

In [29]:
tot_public_floor_space = 376645408.432
new_public_floor_space = 9195076.7802
new_public_ratio = new_public_floor_space / tot_public_floor_space
new_public_ratio

0.0244130860866716

In [34]:
ss_ratio_25 = 0.4
ss_ratio_30 = 0.6
new_public_ratio_ss_25 = fs_gt_500_share * new_public_ratio * ss_ratio_25 * 5
new_public_ratio_ss_30 = new_public_ratio_ss_25 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_35 = new_public_ratio_ss_30 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
new_public_ratio_ss_40 = new_public_ratio_ss_35 + (fs_gt_500_share * ss_ratio_30 * new_public_ratio) * 5
print(new_public_ratio_ss_25, new_public_ratio_ss_30, new_public_ratio_ss_35, new_public_ratio_ss_40)

0.007548526217998858 0.018871315544997148 0.030194104871995437 0.04151689419899372


In [35]:
pr_share = 0.911
serZebShare = pd.Series([
    new_pr_ratio_ss_25 * pr_share + new_public_ratio_ss_25 * (1-pr_share),
    new_pr_ratio_ss_30 * pr_share + new_public_ratio_ss_30 * (1-pr_share),
    new_pr_ratio_ss_35 * pr_share + new_public_ratio_ss_35 * (1-pr_share),
    new_pr_ratio_ss_40 * pr_share + new_public_ratio_ss_40 * (1-pr_share),
], index=[2025, 2030, 2035, 2040])
serZebShare

2025    0.003081
2030    0.014197
2035    0.025313
2040    0.036430
dtype: float64

In [36]:
zebTarget = serFloor * serZebShare * enConUnit
zebTarget

Year
2025    0.005652
2030    0.027337
2035    0.049834
2040    0.071804
dtype: float64

In [37]:
roofTopGen = zebTarget + serRefGen
roofTopGen

2020         NaN
2025    0.023126
2030    0.077973
2035    0.135144
2040    0.175251
dtype: float64

```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="ZEB-Certification-Target">
        <policyType>subsidy</policyType> 
        <market>South Korea</market>
        <constraint year="2020">0.012</constraint>
        <constraint year="2025">0.030</constraint>
        <constraint year="2030">0.096</constraint>
        <constraint year="2035">0.164</constraint>
        <constraint year="2040">0.215</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>
```

# Model Run

```xml
<?xml version="1.0" encoding="UTF-8"?>
<BatchRunner xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:noNamespaceSchemaLocation="batch_runner.xsd">

	<ComponentSet name="Policy">

		<FileSet name="_Reference">
		</FileSet>

		<FileSet name="_CurPol">
			<Value name="zeb">../input/policy/ndc/buildings/zeb_target_constraint.xml</Value>
			<Value name="zeb">../input/policy/ndc/buildings/zeb_target_tech.xml</Value>
		</FileSet>

		<FileSet name="_EnhPol">
			<Value name="zeb">../input/policy/ndc/buildings/zeb_target_constraint_EnhPol.xml</Value>
			<Value name="zeb">../input/policy/ndc/buildings/zeb_target_tech_EnhPol.xml</Value>
		</FileSet>

	</ComponentSet>

</BatchRunner>
```

In [159]:
config_ref_file_path = proj_path / "exe/configuration_ref.xml"
db_path = "../output/database_basexdb_test"
scen_name = "ZEB"
end_year = "2050"
batch_mode = "1"
batch_fn = "batch_test.xml"
config_test_file_path = proj_path / "exe/configuration_test.xml"


tree = ET.parse(config_ref_file_path)
root = tree.getroot()

# set db location
db_loc = root.find(".//Files/Value[@name='xmldb-location']")
db_loc.text = str(db_path)

# set db location
batch_file_name = root.find(".//Files/Value[@name='BatchFileName']")
batch_file_name.text = batch_fn

# set scenario name
scenario_name = root.find(".//Strings/Value[@name='scenarioName']")
scenario_name.text = scen_name

# turn on batch mode
batch_mode = root.find(".//Bools/Value[@name='BatchMode']")
batch_mode.text = "1"

# set stop-year
stop_year = root.find(".//Ints/Value[@name='stop-year']")
stop_year.text = end_year

# save
xml_string = ET.tostring(root, encoding="unicode")
string_to_xml_file(xml_string, config_test_file_path)

XML file '/data/project/tae/gcam-core/exe/configuration_test.xml' created successfully with proper indentation and no extra newlines.


In [161]:
os.chdir("../analyze/script-ndc/buildings")

In [177]:
# Change directory
os.chdir("../../../exe")

# Command to run
command = ["./gcam.exe", "-C", "configuration_test.xml"]

# Run the command and print logs in real-time
try:
    process = subprocess.Popen(
        command, 
        stdout=subprocess.PIPE, 
        stderr=subprocess.PIPE, 
        text=True
    )

    # Print stdout and stderr in real-time
    for line in process.stdout:
        print(line, end="")  # Print each line of stdout
    for line in process.stderr:
        print(line, end="")  # Print each line of stderr

    # Wait for the process to complete
    process.wait()

    # Check return code to see if the process was successful
    if process.returncode == 0:
        print("\nCommand executed successfully!")
    else:
        print("\nCommand failed!")
except Exception as e:
    print(f"An error occurred: {e}")

os.chdir("../analyze/script-ndc/buildings")

This computer software was prepared by Battelle Memorial Institute,
hereinafter the Contractor, under Contract No. DE-AC05-76RL0 1830 with
the Department of Energy (DOE). NEITHER THE GOVERNMENT NOR THE
CONTRACTOR MAKES ANY WARRANTY, EXPRESS OR IMPLIED, OR ASSUMES ANY
LIABILITY FOR THE USE OF THIS SOFTWARE. This notice including this
sentence must appear on any copies of this computer software.

User agrees that the Software will not be shipped, transferred or
exported into any country or used in any manner prohibited by the United
States Export Administration Act or any other applicable export laws,
restrictions or regulations (collectively the 'Export Laws'). Export of
the Software may require some form of license or other authority from
the U.S. Government, and failure to obtain such export control license
may result in criminal liability under U.S. laws. In addition, if the
Software is identified as export controlled items under the Export Laws,
User represents and warrants that Use

In [165]:
os.chdir("../analyze/script-ndc/buildings")

# Model Result Validation

In [178]:
db_file = "database_basexdb_test"
db_path = proj_path / "output"
query_file = db_path / "queries" / "Main_queries.xml"
conn = gcamreader.LocalDBConn(db_path, db_file)
queries = gcamreader.parse_batch_query(query_file)

Database scenarios: ZEB_Reference, ZEB_CurPol, ZEB_EnhPol, ZEB_EnhPol


In [179]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [180]:
q = queries[9]
print(q.title)
dfGen = conn.runQuery(query=q, scenarios=['ZEB_Reference', 'ZEB_CurPol', 'ZEB_EnhPol'], regions=['South Korea'])
dfGen['scenario'] = dfGen['scenario'].str.split(',').str[0]
dfGen.head()

elec gen by gen tech


,Units,scenario,region,subsector,technology,output,Year,value
0,EJ,ZEB_CurPol,South Korea,biomass,biomass (IGCC),electricity,2025,0.000413
1,EJ,ZEB_CurPol,South Korea,biomass,biomass (IGCC),electricity,2030,0.001068
2,EJ,ZEB_CurPol,South Korea,biomass,biomass (IGCC),electricity,2035,0.001997
3,EJ,ZEB_CurPol,South Korea,biomass,biomass (IGCC),electricity,2040,0.003188
4,EJ,ZEB_CurPol,South Korea,biomass,biomass (IGCC),electricity,2045,0.004496


In [181]:
dfGen['technology'].unique()

array(['biomass (IGCC)', 'biomass (conv)', 'coal (IGCC)',
       'coal (conv pul)', 'gas (CC)', 'gas (steam/CT)', 'hydro',
       'Gen_III', 'Gen_II_LWR', 'refined liquids (CC)',
       'refined liquids (steam/CT)', 'rooftop_pv', 'CSP_storage', 'PV',
       'PV_storage', 'wind', 'wind_offshore', 'wind_storage'],
      dtype=object)

In [182]:
dfGen[(dfGen['Year'] >= 2020) & (dfGen['technology'] == 'rooftop_pv')].groupby(['Year', 'scenario'])['value'].sum()

Year  scenario     
2020  ZEB_CurPol       0.002308
      ZEB_EnhPol       0.002308
      ZEB_Reference    0.002308
2025  ZEB_CurPol       0.019953
      ZEB_EnhPol       0.019953
      ZEB_Reference    0.017475
2030  ZEB_CurPol       0.057694
      ZEB_EnhPol       0.093517
      ZEB_Reference    0.050636
2035  ZEB_CurPol       0.097082
      ZEB_EnhPol       0.170335
      ZEB_Reference    0.085309
2040  ZEB_CurPol       0.119796
      ZEB_EnhPol       0.229806
      ZEB_Reference    0.103447
2045  ZEB_CurPol       0.116365
      ZEB_EnhPol       0.124262
      ZEB_Reference    0.115052
2050  ZEB_CurPol       0.120736
      ZEB_EnhPol       0.128216
      ZEB_Reference    0.119492
Name: value, dtype: float64